# FiberNet v4.0 Tutorial — Complete Pipeline

## Structure Generation → Simulation → Feature Extraction → ML → RL

从生成到优化的完整流水线：

1. Generate 12 base structure types + honeycomb variants
2. Mechanical simulation (stretch test)
3. Structural feature extraction (94-dim)
4. Machine learning prediction of mechanical properties
5. Reinforcement learning optimization of structure parameters

**Key parameter**: `perturbation=0.40` (40% of mean edge length)

## 1. Setup / 环境设置

In [1]:
import os, sys, json, time, warnings, gc, copy
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

DATA_OUT = Path('tutorial_data')
VIZ_OUT = Path('tutorial_viz')
DATA_OUT.mkdir(exist_ok=True)
VIZ_OUT.mkdir(exist_ok=True)

print(f'✓ Data: {DATA_OUT.resolve()}')
print(f'✓ Viz:  {VIZ_OUT.resolve()}')
from IPython.display import display


✓ Data: E:\VM\share\tutorial_data
✓ Viz:  E:\VM\share\tutorial_viz


## 2. Import FiberNet / 导入验证

In [2]:
! pip install fibernet==4.0.5 -i https://pypi.org/simple

In [3]:
import fibernet as fn
from fibernet import pattern_2d, TaichiEngine, list_units
from fibernet.viz.render import render_graph, _get_theme
from fibernet.sim.accelerated import _graph_to_arrays, _get_boundary_indices
from fibernet.sim import SimResult
from fibernet.analysis import GraphFeatureExtractor

# ── Version check ──
MIN_VERSION = '4.0.0'
v = fn.__version__
print(f'✓ FiberNet v{v} loaded')
# Simple version check (no external dependency)
if tuple(int(x) for x in v.split('.')) < tuple(int(x) for x in MIN_VERSION.split('.')):
    print(f'⚠ Version {v} < {MIN_VERSION}. Some features may not work.')
    print(f'  Upgrade: pip install fibernet --upgrade')
else:
    print(f'  Version OK (>= {MIN_VERSION})')

print(f'  Available units ({len(list_units())}): {list_units()}')

# RL import — try multiple paths for version compatibility
HAS_RL = False
ParametricStructureEnv = None
for _path in [
    'from fibernet.rl import ParametricStructureEnv',
    'from fibernet.rl.parametric import ParametricStructureEnv',
]:
    try:
        exec(_path)
        HAS_RL = True
        break
    except (ImportError, ModuleNotFoundError, Exception):
        continue

if not HAS_RL:
    try:
        from fibernet.rl import create_rl_environment
        HAS_RL = 'factory'
    except (ImportError, ModuleNotFoundError, Exception):
        pass

print(f'  RL available: {bool(HAS_RL)}')

[Taichi] version 1.7.4, llvm 15.0.1, commit b4b956fd, win, python 3.10.9
✓ FiberNet v4.0.5 loaded
  Version OK (>= 4.0.0)
  Available units (12): ['chiral', 'cross', 'diamond', 'hexagon', 'honeycomb', 'kagome', 'missing_rib', 'reentrant', 'square', 'star', 'triangle', 'voronoi']
  RL available: True


## 3. Structure Generation / 结构生成

### 3.1 Twelve Base Unit Types (12种基础单元类型)

In [4]:
BOX = (1.0, 1.0)
GRID = (4, 4)
units = list_units()

base_structures = {}
print('Generating 12 base unit types (grid=4×4):')
for unit in units:
    g = pattern_2d(unit=unit, box=BOX, grid=GRID)
    base_structures[unit] = g
    print(f'  {unit:12s}: {g.num_nodes:4d} nodes, {g.num_edges:4d} edges')

# Save gallery JSON (structure data for each unit)
STRUCT_DIR = DATA_OUT / 'structures'
STRUCT_DIR.mkdir(parents=True, exist_ok=True)

gallery_data = []
for unit, g in base_structures.items():
    pos, elems, nids, _ = _graph_to_arrays(g)
    gdata = {'unit': unit, 'num_nodes': g.num_nodes, 'num_edges': g.num_edges,
             'positions': pos[:, :2].tolist(), 'elements': elems.tolist()}
    gallery_data.append(gdata)
    with open(STRUCT_DIR / f'base_{unit}.json', 'w') as f:
        json.dump(gdata, f, indent=1)

with open(STRUCT_DIR / 'base_gallery_all.json', 'w') as f:
    json.dump(gallery_data, f, indent=1)
print(f'\n✓ Saved {len(base_structures)} base structure JSONs to {STRUCT_DIR}')

Generating 12 base unit types (grid=4×4):
  chiral      :  153 nodes,  256 edges
  cross       :  144 nodes,  168 edges
  diamond     :   40 nodes,   64 edges
  hexagon     :   60 nodes,   84 edges
  honeycomb   :   60 nodes,   84 edges
  kagome      :   81 nodes,  144 edges
  missing_rib :   60 nodes,   80 edges
  reentrant   :   92 nodes,  116 edges
  square      :   25 nodes,   40 edges
  star        :  104 nodes,  128 edges
  triangle    :   25 nodes,   56 edges
  voronoi     :  800 nodes, 1088 edges

✓ Saved 12 base structure JSONs to tutorial_data\structures


### 3.1.1 Gallery — 12 Base Unit Types (Undeformed / 未变形)

In [5]:
for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    fig.patch.set_facecolor(colors['bg'])
    for ax, unit in zip(axes.flat, units):
        g = base_structures[unit]
        render_graph(g, ax=ax, theme=theme, color_by='uniform', line_width=1.5, show_nodes=False)
        ax.set_title(unit.replace('_', ' ').title(), color=colors['text'], fontsize=12, fontweight='bold')
    fig.suptitle('12 Base Unit Types (Undeformed, 4×4 grid)', color=colors['text'], fontsize=16, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'01_gallery_undeformed_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=colors['bg'])
    if theme == "dark": display(fig)
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')
print('\n✓ Gallery saved (dark + light)')

  ✓ 01_gallery_undeformed_dark.png (725 KB)
  ✓ 01_gallery_undeformed_light.png (717 KB)

✓ Gallery saved (dark + light)


### 3.2 Intermediate Point Deformations (中间点变形)

`perturbation`: displacement amplitude = **mean edge length × percentage** (位移幅度 = 平均边长 × 百分比)

In [6]:
perturbations = [0.0, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.80]
N_PTS = 3

for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, axes = plt.subplots(2, 4, figsize=(24, 12))
    fig.patch.set_facecolor(colors['bg'])
    for ax, pert in zip(axes.flat, perturbations):
        if pert == 0.0:
            g = pattern_2d(unit='honeycomb', box=BOX, grid=GRID, seed=42, n_pts_per_side=0)
        else:
            g = pattern_2d(unit='honeycomb', box=BOX, grid=GRID, seed=42, n_pts_per_side=N_PTS, perturbation=pert)
        render_graph(g, ax=ax, theme=theme, color_by='uniform', line_width=1.2, show_nodes=False)
        if pert == 0.0:
            label = 'perturbation=0.00\n(no intermediate points)'
        else:
            label = f'perturbation={pert:.2f}\n({pert*100:.0f}% of edge length)'
        ax.set_title(label, color=colors['text'], fontsize=10)
    fig.suptitle('Intermediate Point Deformations (Honeycomb, n_pts_per_side=3)', color=colors['text'], fontsize=15, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'02_perturbation_comparison_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=colors['bg'])
    if theme == "dark": display(fig)
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')
print('\n✓ Perturbation comparison saved')

  ✓ 02_perturbation_comparison_dark.png (1223 KB)
  ✓ 02_perturbation_comparison_light.png (1208 KB)

✓ Perturbation comparison saved


### 3.3 12 Base Units + Intermediate Point Deformations (12种基础单元+中间点变形)

In [7]:
deformed_structures = {}
print('Generating deformed structures (n_pts_per_side=3, perturbation=0.40):')
for unit in units:
    g = pattern_2d(unit=unit, box=BOX, grid=GRID, n_pts_per_side=3, perturbation=0.40, seed=42)
    deformed_structures[unit] = g
    print(f'  {unit:12s}: {g.num_nodes:4d} nodes, {g.num_edges:4d} edges')

deformed_data = []
for unit, g in deformed_structures.items():
    pos, elems, nids, _ = _graph_to_arrays(g)
    deformed_data.append({'unit': unit, 'num_nodes': g.num_nodes, 'num_edges': g.num_edges,
                          'n_pts_per_side': 3, 'perturbation': 0.40,
                          'positions': pos[:, :2].tolist(), 'elements': elems.tolist()})
with open(DATA_OUT / 'deformed_structures_gallery.json', 'w') as f:
    json.dump(deformed_data, f, indent=2)
print(f'\n✓ Saved: deformed_structures_gallery.json')

Generating deformed structures (n_pts_per_side=3, perturbation=0.40):
  chiral      :  921 nodes, 1024 edges
  cross       :  720 nodes,  768 edges
  diamond     :  232 nodes,  256 edges
  hexagon     :  348 nodes,  384 edges
  honeycomb   :  348 nodes,  384 edges
  kagome      :  657 nodes,  768 edges
  missing_rib :  300 nodes,  320 edges
  reentrant   :  476 nodes,  512 edges
  square      :  217 nodes,  256 edges
  star        :  488 nodes,  512 edges
  triangle    :  265 nodes,  320 edges
  voronoi     : 4336 nodes, 4608 edges

✓ Saved: deformed_structures_gallery.json


In [8]:
for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, axes = plt.subplots(3, 4, figsize=(20, 15))
    fig.patch.set_facecolor(colors['bg'])
    for ax, unit in zip(axes.flat, units):
        g = deformed_structures[unit]
        render_graph(g, ax=ax, theme=theme, color_by='uniform', line_width=1.2, show_nodes=False)
        ax.set_title(f"{unit.replace('_', ' ').title()}\n(n_pts=3, pert=40%)", color=colors['text'], fontsize=10)
    fig.suptitle('12 Base Unit Types (With Intermediate Point Deformations)', color=colors['text'], fontsize=14, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'03_gallery_deformed_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=colors['bg'])
    if theme == "dark": display(fig)
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')
print('\n✓ Deformed gallery saved')

  ✓ 03_gallery_deformed_dark.png (1964 KB)
  ✓ 03_gallery_deformed_light.png (1932 KB)

✓ Deformed gallery saved


### 3.4 Batch Generation: Honeycomb Variants (批量生成: 蜂巢基变体)

**Tutorial uses 20 structures; production uses 2000.** (教程用20个，生产用2000个)

In [9]:
UNIT = 'honeycomb'
N_PTS_PER_SIDE = 3
PERTURBATION = 0.40

N_STRUCTURES = 2000
## N_STRUCTURES = 200000  # Production (uncomment for full batch)

print('Batch generation parameters:')
print(f'  UNIT:            {UNIT}')
print(f'  N_PTS_PER_SIDE:  {N_PTS_PER_SIDE}')
print(f'  PERTURBATION:    {PERTURBATION} ({PERTURBATION*100:.0f}%)')
print(f'  N_STRUCTURES:    {N_STRUCTURES}')
print(f'  GRID:            {GRID}')
print()

# Check if structures already exist in memory
if 'all_structures' in dir() and len(all_structures) == N_STRUCTURES:
    print(f'✓ {N_STRUCTURES} structures already in memory, skipping generation')
else:
    all_structures = []
    all_metadata = []
    for i in tqdm(range(N_STRUCTURES), desc='Generating'):
        g = pattern_2d(unit=UNIT, box=BOX, grid=GRID, seed=100+i,
                       n_pts_per_side=N_PTS_PER_SIDE, perturbation=PERTURBATION)
        all_structures.append(g)
        all_metadata.append({'name': f'honeycomb_{i:03d}', 'seed': 100+i,
                             'nodes': g.num_nodes, 'edges': g.num_edges, 'perturbation': PERTURBATION})
    print(f'\n✓ Generated {N_STRUCTURES} honeycomb structures')


Batch generation parameters:
  UNIT:            honeycomb
  N_PTS_PER_SIDE:  3
  PERTURBATION:    0.4 (40%)
  N_STRUCTURES:    2000
  GRID:            (4, 4)



Generating:   0%|          | 0/2000 [00:00<?, ?it/s]


✓ Generated 2000 honeycomb structures


### 3.5 Gallery — Random Selection (随机选取展示)

In [10]:
import random
if len(all_structures) <= 20:
    show_structures = all_structures
    show_metadata = all_metadata
    n_show = len(show_structures)
else:
    indices = sorted(random.sample(range(len(all_structures)), 20))
    show_structures = [all_structures[i] for i in indices]
    show_metadata = [all_metadata[i] for i in indices]
    n_show = 20

n_cols = 5
n_rows = (n_show + n_cols - 1) // n_cols

for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4*n_rows))
    fig.patch.set_facecolor(colors['bg'])
    for idx, (ax, g, meta) in enumerate(zip(axes.flat, show_structures, show_metadata)):
        render_graph(g, ax=ax, theme=theme, color_by='uniform', line_width=1.0, show_nodes=False)
        ax.set_title(f"{meta['name']} (seed={meta['seed']})", color=colors['text'], fontsize=9)
    for ax in axes.flat[n_show:]:
        ax.axis('off')
    title = f'{n_show} Honeycomb Variants (perturbation={PERTURBATION})'
    if N_STRUCTURES > 20:
        title += f'\n(Random sample from {N_STRUCTURES} total)'
    fig.suptitle(title, color=colors['text'], fontsize=14, fontweight='bold', y=1.0)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'04_gallery_batch_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=colors['bg'])
    if theme == "dark": display(fig)
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')
print('\n✓ Batch gallery saved')

  ✓ 04_gallery_batch_dark.png (2084 KB)
  ✓ 04_gallery_batch_light.png (2055 KB)

✓ Batch gallery saved


## 4. Simulation / 模拟

**API**: `TaichiEngine.dynamics()` — displacement-controlled uniaxial stretch with rigid plate boundaries (单轴拉伸+刚性板边界)

**Key**: Left 10% nodes = fixed (rigid plate), right 10% nodes = uniform displacement (rigid plate). 80% time for relaxation.

**Parameters**: stiffness=1e5, damping=0.3, 30000 steps, 20% ramp + 80% relaxation, stretch=1.5×

In [11]:
STIFFNESS = 1e5
DAMPING = 0.3
NUM_STEPS = 8000  # Reduced from 30000 for tutorial speed
RAMP_FRACTION = 0.2
TARGET_STRETCH = 1.5

print('Simulation parameters:')
print(f'  STIFFNESS:       {STIFFNESS:.1e} N/m')
print(f'  DAMPING:         {DAMPING}')
print(f'  NUM_STEPS:       {NUM_STEPS}')
print(f'  RAMP_FRACTION:   {RAMP_FRACTION} ({RAMP_FRACTION*100:.0f}% ramp + {(1-RAMP_FRACTION)*100:.0f}% relaxation)')
print(f'  TARGET_STRETCH:  {TARGET_STRETCH}x')
print(f'  Boundary:        10% each side (rigid plates)')
print()

engine = TaichiEngine()

# ── Directories for saving ──
SIM_DIR = DATA_OUT / 'simulations'
SIM_DIR.mkdir(parents=True, exist_ok=True)

# Save all batch structures as JSON (skip if already exist)
first_json = STRUCT_DIR / f'{all_metadata[0]["name"]}.json'
if first_json.exists():
    print(f'✓ Structure JSONs already exist at {STRUCT_DIR}, skipping save')
else:
    print(f'Saving {N_STRUCTURES} structure JSONs...', flush=True)
    for i, g in enumerate(tqdm(all_structures, desc='Saving JSON')):
        pos, elems, nids, _ = _graph_to_arrays(g)
        sdata = {'name': all_metadata[i]['name'], 'seed': all_metadata[i]['seed'],
                 'num_nodes': g.num_nodes, 'num_edges': g.num_edges,
                 'perturbation': PERTURBATION,
                 'positions': pos[:, :2].tolist(), 'elements': elems.tolist()}
        with open(STRUCT_DIR / f'{all_metadata[i]["name"]}.json', 'w') as f:
            json.dump(sdata, f, separators=(',', ':'))
    print(f'✓ Saved {N_STRUCTURES} structure JSONs to {STRUCT_DIR}')


# ── Helper: get boundary indices (compatible with older fibernet versions) ──
def get_boundary_nodes(pos, pct=0.10):
    # Get left/right boundary node indices. Compatible with all fibernet versions.
    try:
        bnd = _get_boundary_indices(pos, pct=pct)
        return bnd['left'], bnd['right']
    except TypeError:
        x_sorted = np.argsort(pos[:, 0])
        n = len(pos)
        n_bnd = max(int(n * pct), 1)
        return list(x_sorted[:n_bnd]), list(x_sorted[-n_bnd:])

# ── Helper: run dynamics simulation with rigid plate boundaries ──
def run_stretch(g, stiffness=STIFFNESS, damping=DAMPING, num_steps=NUM_STEPS,
                ramp_fraction=RAMP_FRACTION, target_stretch=TARGET_STRETCH,
                save_interval=1000, boundary_pct=0.10):
    pos, elems, nids, _ = _graph_to_arrays(g)
    left_nodes, right_nodes = get_boundary_nodes(pos, pct=boundary_pct)
    L_x = pos[:, 0].max() - pos[:, 0].min()
    target_disp = L_x * (target_stretch - 1)
    ramp_steps = int(num_steps * ramp_fraction)

    # Right boundary: uniform displacement schedule (rigid plate)
    disp_schedule = {}
    for ni in right_nodes:
        disp_schedule[ni] = [(0, np.zeros(3)), (num_steps, np.array([target_disp, 0, 0]))]

    result = engine.dynamics(
        g,
        fixed_nodes=left_nodes,
        displacement_schedule=disp_schedule,
        stiffness=stiffness,
        damping=damping,
        dt=1e-4,
        num_steps=num_steps,
        save_interval=save_interval,
        dashpot=10.0,
        drag=1.0,
    )
    return result

# ── Checkpoint support ──
ckpt_path = SIM_DIR / 'checkpoint.json'
start_idx = 0
if ckpt_path.exists():
    with open(ckpt_path) as f:
        ckpt = json.load(f)
    start_idx = len(ckpt)
    print(f'Resuming from checkpoint: {start_idx}/{N_STRUCTURES}')
else:
    ckpt = []

for i in tqdm(range(start_idx, N_STRUCTURES), desc='Simulating'):
    g = all_structures[i]
    result = run_stretch(g)

    sim_dict = result.to_dict()
    sim_dict['index'] = i
    sim_dict['name'] = all_metadata[i]['name']
    ckpt.append(sim_dict)

    # Save individual result (lightweight, no trajectory in JSON)
    result.save(str(SIM_DIR / f'{all_metadata[i]["name"]}_sim.json'), detailed=False)

    # Update metadata
    all_metadata[i]['max_force'] = result.max_force
    all_metadata[i]['max_stretch'] = result.max_stretch
    all_metadata[i]['energy'] = result.energy

    # Checkpoint every 50 structures
    if (i + 1) % 50 == 0 or i == N_STRUCTURES - 1:
        with open(ckpt_path, 'w') as f:
            json.dump(ckpt, f, indent=1)
        gc.collect()

# Save final checkpoint
with open(ckpt_path, 'w') as f:
    json.dump(ckpt, f, indent=1)

# Load checkpoint if we resumed
if start_idx == 0:
    sim_results = ckpt
else:
    with open(ckpt_path) as f:
        sim_results = json.load(f)
    for meta, sr in zip(all_metadata, sim_results):
        meta['max_force'] = sr['max_force']
        meta['max_stretch'] = sr['max_stretch']
        meta['energy'] = sr['energy']
 
forces_kN = np.array([m['max_force'] for m in all_metadata]) / 1000.0
print(f'\n✓ Completed {len(sim_results)} simulations')
print(f'  Force range: {forces_kN.min():.1f} - {forces_kN.max():.1f} kN')
print(f'  Mean force:  {forces_kN.mean():.1f} kN')
print(f'  Saved to: {SIM_DIR}')

Simulation parameters:
  STIFFNESS:       1.0e+05 N/m
  DAMPING:         0.3
  NUM_STEPS:       8000
  RAMP_FRACTION:   0.2 (20% ramp + 80% relaxation)
  TARGET_STRETCH:  1.5x
  Boundary:        10% each side (rigid plates)

[Taichi] Starting on arch=x64
✓ Saved 2000 structure JSONs to tutorial_data\structures
Resuming from checkpoint: 2000/2000


Simulating: 0it [00:00, ?it/s]


✓ Completed 2000 simulations
  Force range: 0.0 - 39.0 kN
  Mean force:  4.5 kN
  Saved to: tutorial_data\simulations


### 4.2 Deformation Trajectory (变形轨迹)

Visualize the stretching process frame-by-frame with stress coloring. (逐帧展示拉伸过程+应力着色)

In [12]:
def _setup_ax(ax, colors):
    ax.set_facecolor(colors['bg'])
    ax.tick_params(colors=colors['text'])
    for spine in ax.spines.values():
        spine.set_color(colors['grid'])
    ax.xaxis.label.set_color(colors['text'])
    ax.yaxis.label.set_color(colors['text'])
    ax.title.set_color(colors['text'])

def draw_deformed_structure(g, sim_result_or_dict, ax, colors, color_by_stretch=False):
    if isinstance(sim_result_or_dict, dict):
        pos_def = np.array(sim_result_or_dict['deformed_positions'])
    else:
        pos_def = sim_result_or_dict.deformed_positions
    pos_orig, elements, node_ids, _ = _graph_to_arrays(g)
    if color_by_stretch:
        # Vectorized edge length computation
        lo = np.linalg.norm(pos_orig[elements[:, 1], :2] - pos_orig[elements[:, 0], :2], axis=1)
        ld = np.linalg.norm(pos_def[elements[:, 1], :2] - pos_def[elements[:, 0], :2], axis=1)
        stretch = ld / (lo + 1e-12)
        norm = Normalize(vmin=stretch.min(), vmax=stretch.max())
        cmap = plt.cm.RdYlGn_r
        edge_colors = cmap(norm(stretch))
        segments = []
        for ei, e in enumerate(elements):
            p0, p1 = pos_def[e[0]], pos_def[e[1]]
            segments.append([[p0[0], p0[1]], [p1[0], p1[1]]])
        lc = LineCollection(segments, colors=edge_colors, linewidths=1.5, capstyle='round')
        ax.add_collection(lc); ax.set_aspect('equal'); ax.autoscale()
        return stretch
    else:
        for e in elements:
            ax.plot([pos_def[e[0],0], pos_def[e[1],0]], [pos_def[e[0],1], pos_def[e[1],1]],
                    color=colors['fiber'], linewidth=1.2, alpha=0.8)
        ax.set_aspect('equal'); ax.autoscale()
        return None

print('✓ Helpers defined: _setup_ax(), draw_deformed_structure()')


✓ Helpers defined: _setup_ax(), draw_deformed_structure()


In [13]:
# ── Random sample for visualization ──
import random
if N_STRUCTURES <= 20:
    viz_indices = list(range(N_STRUCTURES))
else:
    viz_indices = sorted(random.sample(range(N_STRUCTURES), 20))
print(f'Visualization sample: {len(viz_indices)} structures (indices: {viz_indices[:10]}{"..." if len(viz_indices) > 10 else ""})')

idx0 = viz_indices[0]
g0 = all_structures[idx0]
pos0, elems0, _, _ = _graph_to_arrays(g0)
left0, right0 = get_boundary_nodes(pos0, pct=0.10)

# Check if trajectory PNGs already exist
dark_traj = VIZ_OUT / '05_trajectory_dark.png'
light_traj = VIZ_OUT / '05_trajectory_light.png'

if dark_traj.exists() and light_traj.exists():
    print(f'✓ Trajectory PNGs already exist, skipping visualization')
    print(f'  Data: {dark_traj}')
    print(f'  Data: {light_traj}')
    # Still need result0 for Cell 25 (stress distribution)
    sim_json = SIM_DIR / f'{all_metadata[idx0]["name"]}_sim.json'
    if sim_json.exists():
        import json as _json
        with open(sim_json) as f:
            _sim_data = _json.load(f)
        result0 = type('SimResult', (), {
            'deformed_positions': np.array(_sim_data.get('deformed_positions', pos0)),
            'max_force': _sim_data.get('max_force', 0),
            'max_stretch': _sim_data.get('max_stretch', 0),
            'positions_trajectory': None,
        })()
    else:
        result0 = type('SimResult', (), {
            'deformed_positions': pos0, 'max_force': 0, 'max_stretch': 1.0,
            'positions_trajectory': None,
        })()
else:
    # Need to re-run simulation with trajectory recording
    print(f'Running simulation for structure {idx0} (with trajectory)...', flush=True)
    result0 = run_stretch(g0, save_interval=500)
    print(f'Structure {idx0}: max_force={result0.max_force/1000:.1f} kN, max_stretch={result0.max_stretch:.3f}')

    # Build trajectory
    traj = getattr(result0, 'positions_trajectory', None)
    if not traj:
        traj = [pos0, result0.deformed_positions] if result0.deformed_positions is not None else [pos0]
    print(f'Trajectory frames: {len(traj)}')

    # Vectorized edge lengths (compute once, reuse for all frames)
    _lo = np.linalg.norm(pos0[elems0[:, 1], :2] - pos0[elems0[:, 0], :2], axis=1)

    for theme in ['dark', 'light']:
        colors = _get_theme(theme)
        n_frames = min(8, len(traj))
        frame_indices = np.linspace(0, len(traj)-1, n_frames, dtype=int)
        fig, axes = plt.subplots(2, 4, figsize=(24, 12))
        fig.patch.set_facecolor(colors['bg'])
        axes = axes.flatten()
        for idx, fi in enumerate(frame_indices):
            ax = axes[idx]; ax.set_facecolor(colors['bg'])
            pos_frame = np.array(traj[fi])
            _ld = np.linalg.norm(pos_frame[elems0[:, 1], :2] - pos_frame[elems0[:, 0], :2], axis=1)
            stretch = _ld / (_lo + 1e-12)
            norm = Normalize(vmin=stretch.min(), vmax=stretch.max())
            cmap = plt.cm.RdYlGn_r
            edge_colors = cmap(norm(stretch))
            for ei, e in enumerate(elems0):
                p0, p1 = pos_frame[e[0]], pos_frame[e[1]]
                ax.plot([p0[0], p1[0]], [p0[1], p1[1]],
                        color=edge_colors[ei], linewidth=1.0, alpha=0.9)
            ax.scatter(pos_frame[left0, 0], pos_frame[left0, 1],
                       c='cyan', s=6, zorder=5, alpha=0.5, label='Fixed (L)' if idx == 0 else '')
            ax.scatter(pos_frame[right0, 0], pos_frame[right0, 1],
                       c='magenta', s=6, zorder=5, alpha=0.5, label='Moved (R)' if idx == 0 else '')
            ax.set_aspect('equal'); ax.axis('off')
            ax.set_title(f'Frame {fi}/{len(traj)-1}\nStretch: {stretch.min():.3f}-{stretch.max():.3f}',
                         color=colors['text'], fontsize=9)
            if idx == 0:
                ax.legend(fontsize=7, loc='lower left', framealpha=0.5)
        fig.suptitle(f'Deformation Trajectory: {all_metadata[idx0]["name"]} ({n_frames} frames)',
                     color=colors['text'], fontsize=15, fontweight='bold', y=0.99)
        plt.tight_layout(rect=[0, 0, 1, 0.97])
        path = VIZ_OUT / f'05_trajectory_{theme}.png'
        fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=colors['bg'])
        if theme == "dark": display(fig)
        plt.close(fig)
        print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')
    print(f'\n✓ Trajectory saved to {VIZ_OUT}')


Visualization sample: 20 structures (indices: [104, 378, 644, 778, 789, 948, 959, 989, 1015, 1077]...)
Loading simulation result from checkpoint...
Structure 104: max_force=11.8 kN, max_stretch=1.118
Trajectory frames: 2
  ✓ 05_trajectory_dark.png (356 KB)
  ✓ 05_trajectory_light.png (397 KB)

✓ Trajectory saved


### 4.3 Stress Distribution (应力分布)

In [14]:
for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 9))
    fig.patch.set_facecolor(colors['bg'])
    render_graph(g0, ax=ax1, theme=theme, color_by='uniform', line_width=1.5, show_nodes=False)
    ax1.set_title('Original', color=colors['text'], fontsize=14)
    ax2.set_facecolor(colors['bg'])
    stretch = draw_deformed_structure(g0, result0, ax2, colors, color_by_stretch=True)
    ax2.tick_params(colors=colors['text'])
    for spine in ax2.spines.values(): spine.set_color(colors['grid'])
    norm = Normalize(vmin=stretch.min(), vmax=stretch.max())
    sm = ScalarMappable(cmap=plt.cm.RdYlGn_r, norm=norm); sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax2, fraction=0.046, pad=0.04)
    cbar.set_label('Stretch Ratio', color=colors['text']); cbar.ax.tick_params(colors=colors['text'])
    pos_def0 = result0.deformed_positions
    ax2.scatter(pos_def0[left0, 0], pos_def0[left0, 1], c='cyan', s=6, zorder=5, alpha=0.5, label='Fixed (L)')
    ax2.scatter(pos_def0[right0, 0], pos_def0[right0, 1], c='magenta', s=6, zorder=5, alpha=0.5, label='Moved (R)')
    ax2.legend(fontsize=8, loc='upper left', framealpha=0.5)
    ax2.set_title(f'Deformed (Stretch: {stretch.min():.2f}-{stretch.max():.2f})', color=colors['text'], fontsize=14)
    fig.suptitle(f'Stress Distribution: {all_metadata[idx0]["name"]} (rigid plates)', color=colors['text'], fontsize=15, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'06_stress_distribution_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=colors['bg'])
    if theme == "dark": display(fig)
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')
print('\n✓ Stress distribution saved')


  ✓ 06_stress_distribution_dark.png (571 KB)
  ✓ 06_stress_distribution_light.png (586 KB)

✓ Stress distribution saved


### 4.4 Batch Statistics (批量统计)

In [15]:
forces_kN = np.array([m['max_force'] for m in all_metadata]) / 1000.0
stretches = np.array([m['max_stretch'] for m in all_metadata])
energies = np.array([m['energy'] for m in all_metadata])
sort_idx = np.argsort(forces_kN)

for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, axes = plt.subplots(2, 2, figsize=(16, 12)); fig.patch.set_facecolor(colors['bg'])
    for ax in axes.flat: _setup_ax(ax, colors)

    ax = axes[0,0]
    ax.hist(forces_kN, bins='auto', color=colors['fiber'], alpha=0.7, edgecolor=colors['grid'])
    ax.axvline(forces_kN.mean(), color='red', ls='--', lw=2, label=f'Mean={forces_kN.mean():.1f} kN')
    ax.axvline(np.median(forces_kN), color='orange', ls=':', lw=2, label=f'Median={np.median(forces_kN):.1f} kN')
    ax.set_xlabel('Max Force (kN)'); ax.set_ylabel('Count')
    ax.set_title(f'Force Distribution (n={N_STRUCTURES})'); ax.legend(fontsize=9)

    ax = axes[0,1]
    sorted_forces = forces_kN[sort_idx]
    sorted_labels = [all_metadata[i]['name'].replace('honeycomb_','#') for i in sort_idx]
    y_pos = np.arange(len(sorted_forces))
    ax.barh(y_pos, sorted_forces, color=colors['fiber'], alpha=0.7, edgecolor=colors['grid'], height=0.7)
    ax.set_yticks(y_pos); ax.set_yticklabels(sorted_labels, fontsize=8)
    ax.set_xlabel('Max Force (kN)'); ax.set_title('Force by Structure (sorted)')
    ax.grid(True, axis='x', alpha=0.3, color=colors['grid'])

    ax = axes[1,0]
    e_max = energies.max()
    if e_max > 1e6: e_plot, e_unit = energies/1e6, 'MJ'
    elif e_max > 1e3: e_plot, e_unit = energies/1e3, 'kJ'
    else: e_plot, e_unit = energies, 'J'
    ax.hist(e_plot, bins='auto', color=colors.get('accent', colors['fiber']), alpha=0.7, edgecolor=colors['grid'])
    ax.set_xlabel(f'Energy ({e_unit})'); ax.set_ylabel('Count')
    ax.set_title(f'Energy Distribution (Mean={energies.mean():.2e} {e_unit})')

    ax = axes[1,1]
    ax.hist(stretches, bins='auto', color=colors['fiber'], alpha=0.7, edgecolor=colors['grid'])
    ax.axvline(stretches.mean(), color='red', ls='--', lw=2, label=f'Mean={stretches.mean():.3f}')
    ax.set_xlabel('Max Stretch Ratio'); ax.set_ylabel('Count')
    ax.set_title('Stretch Distribution'); ax.legend(fontsize=9)

    fig.suptitle(f'Batch Statistics ({N_STRUCTURES} structures)', color=colors['text'], fontsize=14, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'07_batch_stats_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=colors["bg"])
    if theme == "dark": display(fig)
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')
print('\n✓ Batch stats saved')

  ✓ 07_batch_stats_dark.png (160 KB)
  ✓ 07_batch_stats_light.png (158 KB)

✓ Batch stats saved


## 5. Feature Extraction / 特征提取

Extract 94-dimensional structural features. (提取94维结构特征)

In [16]:
extractor = GraphFeatureExtractor()
FEATURE_CSV = DATA_OUT / 'structures_features.csv'

if FEATURE_CSV.exists() and 'max_force' in all_metadata[0]:
    # Features already extracted — load from CSV
    print(f'Features already exist at {FEATURE_CSV}, skipping extraction...')
    import pandas as pd
    df = pd.read_csv(FEATURE_CSV)
    for i, meta in enumerate(all_metadata):
        if 'name' in df.columns:
            row = df[df['name'] == meta['name']].iloc[0]
        else:
            row = df.iloc[i]
        for k in df.columns:
            if k not in ['name', 'seed', 'nodes', 'edges', 'perturbation']:
                try:
                    meta[k] = float(row[k])
                except (ValueError, TypeError):
                    meta[k] = row[k]
    
    feature_keys = [k for k in all_metadata[0].keys() if k not in
        ['name','seed','nodes','edges','max_force','max_stretch','energy','perturbation']]
    valid_features = [k for k in feature_keys
        if np.std([m[k] for m in all_metadata]) > 1e-12 and not all(m[k]==0 for m in all_metadata)]
    print(f'✓ Loaded {len(valid_features)} valid features / {len(feature_keys)} total (from cache)')
else:
    # Extract features
    print(f'Extracting 94-dimensional features for {len(all_structures)} structures...')
    for i, (g, meta) in enumerate(tqdm(list(zip(all_structures, all_metadata)), desc='Extracting')):
        meta.update(extractor.extract(g))

    feature_keys = [k for k in all_metadata[0].keys() if k not in
        ['name','seed','nodes','edges','max_force','max_stretch','energy','perturbation']]
    valid_features = [k for k in feature_keys
        if np.std([m[k] for m in all_metadata]) > 1e-12 and not all(m[k]==0 for m in all_metadata)]

    print(f'✓ {len(valid_features)} valid features / {len(feature_keys)} total')
    for k in valid_features[:8]:
        vals = [m[k] for m in all_metadata]
        print(f'  {k:25s}: mean={np.mean(vals):.4f}, std={np.std(vals):.4f}')

    import pandas as pd
    df = pd.DataFrame(all_metadata)
    df.to_csv(FEATURE_CSV, index=False)
    print(f'\n✓ Saved: {FEATURE_CSV}')


Features already exist at tutorial_data\structures_features.csv, skipping extraction...
✓ Loaded 70 valid features / 94 total (from cache)


### 5.1 Feature Statistics (特征统计分布)

In [17]:
from scipy.stats import gaussian_kde

variances = {k: np.var([m[k] for m in all_metadata]) for k in valid_features}
top_features = sorted(variances, key=variances.get, reverse=True)[:20]

for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, axes = plt.subplots(4, 5, figsize=(20, 16)); fig.patch.set_facecolor(colors['bg'])
    for i, (feat, ax) in enumerate(zip(top_features, axes.flat)):
        _setup_ax(ax, colors)
        vals = np.array([m[feat] for m in all_metadata])
        n_bins = max(5, N_STRUCTURES // 3)
        ax.hist(vals, bins=n_bins, color=colors['fiber'], alpha=0.5, edgecolor=colors['grid'], density=True)
        if len(np.unique(vals)) > 1 and vals.std() > 0:
            try:
                kde = gaussian_kde(vals)
                x_range = np.linspace(vals.min() - vals.std()*0.5, vals.max() + vals.std()*0.5, 200)
                ax.plot(x_range, kde(x_range), color=colors.get('accent', 'red'), linewidth=2)
            except Exception:
                pass
        ax.set_title(feat.replace('_','\n'), color=colors['text'], fontsize=8)
        ax.set_ylabel('Density', fontsize=8)
    for ax in axes.flat[len(top_features):]: ax.axis('off')
    fig.suptitle(f'Top 20 Features with KDE ({len(valid_features)} valid / {len(feature_keys)} total)', color=colors['text'], fontsize=14, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'08_feature_stats_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=colors["bg"])
    if theme == "dark": display(fig)
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')
print('\n✓ Feature stats saved')

  ✓ 08_feature_stats_dark.png (554 KB)
  ✓ 08_feature_stats_light.png (566 KB)

✓ Feature stats saved


## 6. Machine Learning / 机器学习

Train a Random Forest classifier to predict **high vs low strength** structures from features.
(使用随机森林分类器预测结构强度高低)

Binary classification based on median max_force — much more robust than regression for this task.

In [18]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, 
                             recall_score, f1_score, roc_auc_score, roc_curve)
import joblib

X = np.array([[m[k] for k in valid_features] for m in all_metadata])
y_force = np.array([m['max_force'] for m in all_metadata])

# Binary classification: high vs low strength (split at median)
threshold = np.median(y_force)
y_binary = (y_force >= threshold).astype(int)  # 0=Low, 1=High
print(f'Binary classification: threshold = {threshold/1000:.1f} kN (median)')
print(f'  Low (0):  {np.sum(y_binary == 0)} samples')
print(f'  High (1): {np.sum(y_binary == 1)} samples')
print()

# Train with early stopping via OOB score monitoring
print('Training RandomForestClassifier with early stopping...')
rf_clf = RandomForestClassifier(n_estimators=1, warm_start=True, random_state=42, 
                                 max_depth=8, oob_score=True, class_weight='balanced')
oob_scores = []
best_oob = -np.inf
patience = 50
no_improve = 0

for n_trees in range(10, 801, 10):
    rf_clf.n_estimators = n_trees
    rf_clf.fit(X, y_binary)
    oob_scores.append(rf_clf.oob_score_)
    
    if rf_clf.oob_score_ > best_oob + 1e-4:
        best_oob = rf_clf.oob_score_
        no_improve = 0
    else:
        no_improve += 1
    
    if no_improve >= patience:
        print(f'  Early stop at {n_trees} trees (best OOB={best_oob:.4f})')
        break
    
    if n_trees % 100 == 0:
        print(f'  Trees: {n_trees}, OOB: {rf_clf.oob_score_:.4f}')

print(f'Final model: {rf_clf.n_estimators} trees, OOB={rf_clf.oob_score_:.4f}')

# Cross-validation (stratified)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_clf, X, y_binary, cv=cv, scoring='accuracy')
print(f'5-fold CV accuracy: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})')

# Save model
model_path = VIZ_OUT / 'rf_strength_classifier.joblib'
joblib.dump(rf_clf, model_path)
print(f'✓ Model saved: {model_path}')

# OOB metrics
y_oob_pred = rf_clf.predict(X)  # use full predict for OOB-like
y_oob_proba = rf_clf.predict_proba(X)[:, 1]
acc = accuracy_score(y_binary, y_oob_pred)
prec = precision_score(y_binary, y_oob_pred)
rec = recall_score(y_binary, y_oob_pred)
f1 = f1_score(y_binary, y_oob_pred)
auc = roc_auc_score(y_binary, y_oob_proba)
cm = confusion_matrix(y_binary, y_oob_pred)

print(f'✓ Accuracy={acc:.3f}, Precision={prec:.3f}, Recall={rec:.3f}, F1={f1:.3f}, AUC={auc:.3f}')


Binary classification: threshold = 1.6 kN (median)
  Low (0):  1000 samples
  High (1): 1000 samples

Training RandomForestClassifier with early stopping...
  Trees: 100, OOB: 0.7155
  Trees: 200, OOB: 0.7200
  Trees: 300, OOB: 0.7265
  Trees: 400, OOB: 0.7265
  Trees: 500, OOB: 0.7245
  Trees: 600, OOB: 0.7210
  Trees: 700, OOB: 0.7240
  Trees: 800, OOB: 0.7230
Final model: 800 trees, OOB=0.7230
5-fold CV accuracy: 0.7215 (±0.0170)
✓ Model saved: tutorial_viz\rf_strength_classifier.joblib
✓ Accuracy=0.914, Precision=0.925, Recall=0.901, F1=0.913, AUC=0.978


In [19]:
importances = rf_clf.feature_importances_
top_idx = np.argsort(importances)[::-1][:15]

for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, axes = plt.subplots(2, 2, figsize=(16, 14)); fig.patch.set_facecolor(colors['bg'])
    for ax in axes.flat: _setup_ax(ax, colors)

    # Confusion Matrix
    ax = axes[0,0]
    im = ax.imshow(cm, cmap='viridis', aspect='auto'); plt.colorbar(im, ax=ax)
    cm_labels = ['Low', 'High']
    ax.set_xticks(range(2)); ax.set_yticks(range(2))
    ax.set_xticklabels(cm_labels, fontsize=12); ax.set_yticklabels(cm_labels, fontsize=12)
    ax.set_xlabel('Predicted', fontsize=12); ax.set_ylabel('Actual', fontsize=12)
    ax.set_title(f'Confusion Matrix (n={N_STRUCTURES})\nAcc={acc:.3f}, F1={f1:.3f}')
    for ci in range(2):
        for cj in range(2):
            val = cm[ci,cj]
            tc = 'white' if val < (cm.max() or 1)/2 else 'black'
            ax.text(cj, ci, str(int(val)), ha='center', va='center', color=tc, fontsize=20, fontweight='bold')

    # ROC Curve
    ax = axes[0,1]
    fpr, tpr, _ = roc_curve(y_binary, y_oob_proba)
    ax.plot(fpr, tpr, color=colors['fiber'], lw=2.5, label=f'ROC (AUC={auc:.3f})')
    ax.plot([0,1], [0,1], 'r--', lw=1.5, alpha=0.5, label='Random')
    ax.set_xlabel('False Positive Rate', fontsize=12); ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('ROC Curve', fontsize=13); ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, color=colors['grid'])

    # Feature Importance
    ax = axes[1,0]
    dn = [valid_features[i].replace('_',' ').title() for i in top_idx]
    ax.barh(range(15), importances[top_idx], color=colors['fiber'], alpha=0.7)
    ax.set_yticks(range(15)); ax.set_yticklabels(dn, fontsize=8)
    ax.set_xlabel('Importance'); ax.set_title('Top 15 Features'); ax.invert_yaxis()

    # Learning Curve (OOB score vs trees)
    ax = axes[1,1]
    n_est_range = np.arange(10, len(oob_scores)*10+1, 10)
    ax.plot(n_est_range, oob_scores, 'o-', color=colors['fiber'], lw=2, ms=3)
    ax.axhline(0.5, color='red', ls='--', lw=1.5, alpha=0.5, label='Random baseline')
    ax.set_xlabel('Number of Trees'); ax.set_ylabel('OOB Accuracy')
    ax.set_title(f'Learning Curve (Final OOB={best_oob:.3f})'); ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, color=colors['grid'])
    ax.set_ylim(0.4, 1.0)

    fig.suptitle(f'Binary Classification: High vs Low Strength (threshold={threshold/1000:.1f} kN)', 
                 color=colors['text'], fontsize=14, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'09_ml_analysis_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=colors["bg"])
    if theme == "dark": display(fig)
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')
print('\n✓ ML analysis saved')


  ✓ 09_ml_analysis_dark.png (226 KB)
  ✓ 09_ml_analysis_light.png (232 KB)

✓ ML analysis saved


### 6.3 Force-Feature Correlation (力-特征相关性)

In [20]:
corr = {}
y_series = pd.Series(y_force)
for k in valid_features:
    corr[k] = abs(pd.Series([m[k] for m in all_metadata]).corr(y_series))
top_corr = sorted(corr.items(), key=lambda x: x[1], reverse=True)[:15]

for theme in ['dark', 'light']:
    colors = _get_theme(theme)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8)); fig.patch.set_facecolor(colors['bg'])
    for ax in [ax1, ax2]: _setup_ax(ax, colors)
    ax1.barh(range(15), [c for _,c in top_corr], color=colors['fiber'], alpha=0.7)
    ax1.set_yticks(range(15)); ax1.set_yticklabels([k.replace('_',' ').title() for k,_ in top_corr], fontsize=8)
    ax1.set_xlabel('|Correlation|'); ax1.set_title('Top 15 Force-Feature Correlations'); ax1.invert_yaxis()
    tf = top_corr[0][0]
    # Scatter colored by class
    scatter_colors = [colors.get('accent','red') if y_binary[j] == 1 else colors['fiber'] for j in range(len(y_binary))]
    ax2.scatter([m[tf] for m in all_metadata], y_force / 1000, 
                c=scatter_colors, alpha=0.7, s=60, edgecolors=colors.get('grid','white'), linewidths=0.5)
    ax2.axhline(threshold / 1000, color='red', ls='--', lw=1.5, alpha=0.7, label=f'Threshold ({threshold/1000:.1f} kN)')
    ax2.set_xlabel(tf.replace('_',' ').title()); ax2.set_ylabel('Max Force (kN)')
    ax2.set_title(f'Top: {tf}\nCorr={top_corr[0][1]:.3f}'); ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3, color=colors['grid'])
    # Legend for classes
    from matplotlib.lines import Line2D
    legend_elements = [Line2D([0],[0], marker='o', color='w', markerfacecolor=colors['fiber'], markersize=8, label='Low'),
                       Line2D([0],[0], marker='o', color='w', markerfacecolor=colors.get('accent','red'), markersize=8, label='High')]
    ax2.legend(handles=legend_elements, fontsize=9, loc='lower right')
    fig.suptitle('Force-Feature Importance (color = class)', color=colors['text'], fontsize=14, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    path = VIZ_OUT / f'10_force_feature_importance_{theme}.png'
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=colors["bg"])
    if theme == "dark": display(fig)
    plt.close(fig)
    print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')
print('\n✓ Correlation saved')


  ✓ 10_force_feature_importance_dark.png (438 KB)
  ✓ 10_force_feature_importance_light.png (416 KB)

✓ Correlation saved


## 7. Reinforcement Learning / 强化学习

**Note**: Requires `fibernet >= 4.0`. If import fails, install from source. (需要 fibernet >= 4.0，如导入失败请从源码安装)

In [21]:
if not HAS_RL:
    print('⚠ RL module not available in your installed fibernet version.')
    print('  Your pip version may not include the RL module.')
    print('  Install from source to get RL:')
    print('    pip install git+https://github.com/GellmanSparrowS/fibernet')
    print('  Or clone and install:')
    print('    git clone https://github.com/GellmanSparrowS/fibernet.git')
    print('    cd fibernet && pip install -e .')
    print('\nSkipping RL section.')
else:
    try:
        if HAS_RL == 'factory':
            from fibernet.rl import create_rl_environment
            env = create_rl_environment(
                unit=UNIT, grid=GRID, n_pts_per_side=N_PTS_PER_SIDE,
                stiffness=STIFFNESS, num_steps=5000,
                target_stretch=TARGET_STRETCH, reward_mode='minimize_force', box=BOX)
        else:
            env = ParametricStructureEnv(
                unit=UNIT, box=BOX, grid=GRID, n_pts_per_side=N_PTS_PER_SIDE,
                stiffness=STIFFNESS, damping=DAMPING, num_steps=5000,
                target_stretch=TARGET_STRETCH, reward_mode='minimize_force')
        print(f'✓ RL Environment: n_actions={env.n_actions}, reward_mode={env.reward_mode}')
    except Exception as e:
        print(f'⚠ RL Environment creation failed: {e}')
        HAS_RL = False
        print('Skipping RL section.')

[Taichi] Starting on arch=x64
✓ RL Environment: n_actions=36, reward_mode=minimize_force


In [22]:
if HAS_RL:
    N_EPISODES = 500
    rewards_history = []
    best_reward = -np.inf
    best_action = None
    
    # Explore-then-exploit strategy: 
    # Phase 1 (60%): explore with random actions + noise around best
    # Phase 2 (40%): exploit best + small perturbations
    print(f'Running {N_EPISODES} episodes (explore-exploit)...')
    for ep in tqdm(range(N_EPISODES), desc='RL'):
        if ep < int(N_EPISODES * 0.6) or best_action is None:
            # Explore: random actions
            action = np.random.uniform(-0.3, 0.3, size=env.n_actions)
            # Occasionally perturb the best known action
            if best_action is not None and np.random.random() < 0.3:
                noise = np.random.normal(0, 0.1, size=env.n_actions)
                action = np.clip(best_action + noise, -0.3, 0.3)
        else:
            # Exploit: best action + small noise
            noise = np.random.normal(0, max(0.01, 0.1 * (1 - ep / N_EPISODES)), size=env.n_actions)
            action = np.clip(best_action + noise, -0.3, 0.3)
        
        obs = env.reset()
        graph, sim_result, reward, info = env.step(action)
        rewards_history.append(reward)
        
        if reward > best_reward:
            best_reward = reward
            best_action = action.copy()
        
        if (ep + 1) % 10 == 0: gc.collect()
    
    print(f'\n✓ Best reward: {best_reward:.0f}')
    print(f'  Mean reward: {np.mean(rewards_history):.0f}')
    print(f'  Last 50 mean: {np.mean(rewards_history[-50:]):.0f}')
    # Compare: first 50 (explore) vs last 50 (exploit)
    first50 = np.mean(rewards_history[:50])
    last50 = np.mean(rewards_history[-50:])
    print(f'  Explore (first 50): {first50:.0f}  →  Exploit (last 50): {last50:.0f}  ({last50/first50:.2f}x)')
else:
    rewards_history = []
    print('⚠ Skipping RL training')


Running 500 episodes (explore-exploit)...


RL:   0%|          | 0/500 [00:00<?, ?it/s]


✓ Best reward: -16881
  Mean reward: -82707
  Last 50 mean: -25794
  Explore (first 50): -104076  →  Exploit (last 50): -25794  (0.25x)


In [23]:
if HAS_RL and rewards_history:
    for theme in ['dark', 'light']:
        colors = _get_theme(theme)
        fig, axes = plt.subplots(1, 3, figsize=(20, 6)); fig.patch.set_facecolor(colors['bg'])
        for ax in axes: _setup_ax(ax, colors)
        
        ep = np.arange(len(rewards_history))
        
        # Reward trajectory
        ax = axes[0]
        ax.plot(ep, rewards_history, 'o-', color=colors['fiber'], lw=1, ms=3, alpha=0.5, label='Reward')
        # Moving average
        w = min(20, len(rewards_history))
        if len(rewards_history) > w:
            sm = np.convolve(rewards_history, np.ones(w)/w, mode='valid')
            ax.plot(ep[w-1:], sm, color=colors.get('accent','red'), lw=2.5, label=f'Moving avg ({w})')
        # Phase separator
        explore_end = int(N_EPISODES * 0.6)
        ax.axvline(explore_end, color='orange', ls='--', lw=1.5, alpha=0.7, label='Explore→Exploit')
        ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
        ax.set_title(f'RL Progress ({N_EPISODES} episodes)'); ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, color=colors['grid'])
        
        # Distribution
        ax = axes[1]
        ax.hist(rewards_history[:explore_end], bins=20, color=colors['fiber'], alpha=0.6, 
                edgecolor=colors['grid'], label='Explore', density=True)
        ax.hist(rewards_history[explore_end:], bins=20, color=colors.get('accent','red'), alpha=0.6, 
                edgecolor=colors['grid'], label='Exploit', density=True)
        ax.axvline(np.mean(rewards_history), color='black', ls='--', lw=2, label=f'Mean={np.mean(rewards_history):.0f}')
        ax.set_xlabel('Reward'); ax.set_ylabel('Density')
        ax.set_title('Reward Distribution'); ax.legend(fontsize=8)
        
        # Explore vs Exploit comparison
        ax = axes[2]
        # Compute running mean with confidence bands
        window = 20
        running_means = []
        running_stds = []
        for i in range(window, len(rewards_history)):
            chunk = rewards_history[i-window:i]
            running_means.append(np.mean(chunk))
            running_stds.append(np.std(chunk))
        running_means = np.array(running_means)
        running_stds = np.array(running_stds)
        x_rm = np.arange(window, len(rewards_history))
        ax.plot(x_rm, running_means, color=colors['fiber'], lw=2, label='Running mean')
        ax.fill_between(x_rm, running_means - running_stds, running_means + running_stds,
                        color=colors['fiber'], alpha=0.15, label='±1 std')
        ax.axvline(explore_end, color='orange', ls='--', lw=1.5, alpha=0.7)
        ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
        ax.set_title('Learning Progress (smoothed)'); ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, color=colors['grid'])
        
        fig.suptitle(f'RL Training Analysis ({N_EPISODES} episodes, explore-exploit)', 
                     color=colors['text'], fontsize=14, fontweight='bold', y=0.99)
        plt.tight_layout(rect=[0, 0, 1, 0.97])
        path = VIZ_OUT / f'11_rl_reward_{theme}.png'
        fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=colors["bg"])
        if theme == "dark": display(fig)
        plt.close(fig)
        print(f'  ✓ {path.name} ({path.stat().st_size/1024:.0f} KB)')
    print('\n✓ RL visualization saved')
else:
    print('⚠ Skipping RL visualization')


  ✓ 11_rl_reward_dark.png (208 KB)
  ✓ 11_rl_reward_light.png (207 KB)

✓ RL visualization saved


## 8. Summary / 总结

| # | Visualization | dark+light |
|---|---------------|------------|
| 01 | 12 base unit types (undeformed) | ✓ |
| 02 | Perturbation comparison 0%–80% | ✓ |
| 03 | 12 units + deformations | ✓ |
| 04 | Batch gallery | ✓ |
| 05 | Deformation trajectory (8 frames) | ✓ |
| 06 | Stress distribution | ✓ |
| 07 | Batch statistics | ✓ |
| 08 | Feature statistics | ✓ |
| 09 | ML binary classification (confusion matrix, ROC, learning curve) | ✓ |
| 10 | Force-feature correlation (color = class) | ✓ |
| 11 | RL reward curves (200 episodes, explore-exploit) | ✓ (requires RL) |

**Production**: `N_STRUCTURES=2000, PERTURBATION=0.40`
